# DA016-parent A/B/C/D result diagnosis

## TL;DR

D used the intended M2D-LIF train/val and test annotations. It does not improve the model: test mAP50-95 is 0.664, which is 0.003 below A, 0.004 below B, and 0.006 below C. Prompt quality is at least as good as B/C, so the regression is not caused by Prompt failing to learn; the P3-only residual-to-detection mapping is the failing component in this run.

## Context & Methods

D inherits B's P3/P4 Prompt auxiliary supervision, adds C's zero-initialized bounded residual at P3 only, leaves P4 auxiliary-only, and leaves P5 unchanged. A/B/C/D use seed 0 and 100 epochs, but deterministic training is disabled, so single-run differences are descriptive rather than confidence-bounded.

### Key Assumptions

The evaluator's printed values are treated as rounded to roughly 0.001. Validation comparisons use exact values from results.csv.

In [1]:
from pathlib import Path
from collections import Counter
from pprint import pprint
import csv, re, statistics

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'runs/DroneVehicle_OBB_FusionTransfer').is_dir())
BASE = ROOT / 'runs/DroneVehicle_OBB_FusionTransfer'
RUNS = {
    'A': BASE / 'DA016-A_OriginalSemanticDisagreementLAF-P34_M2DLIFLabels_v1',
    'B': BASE / 'DA016-B_DualReliabilityPromptAuxOnly-P34_M2DLIFLabels_v12',
    'C': BASE / 'DA016-C_DualReliabilityPrompt-ZeroInitBoundedResidual-P34_M2DLIFLabels_v12',
    'D': BASE / 'DA016-D_DualReliabilityPrompt-P3ZeroInitBoundedResidual-P4AuxOnly_M2DLIFLabels_v1',
}
for name, path in RUNS.items():
    assert (path / 'results.csv').is_file()
    assert (path / 'weights/best.pt').is_file()
    assert (path / 'test_m2dlif/test.txt').is_file()

## Data

The first check verifies the source label roots, RGB/IR/label stem integrity, annotation counts, and the D test evaluator's declared paths.

In [2]:
DATA_ROOT = Path('/media/biiteam/新加卷1/biiteam/MCONG/datasets')
TRAIN_LABEL_ROOT = DATA_ROOT / 'M2D-LIFlabels/DroneVehicle_train_val_labels/labels'
SOURCE_ROOT = DATA_ROOT / 'DroneVehicle_twostream_3'
TEST_LABEL_ROOT = DATA_ROOT / 'M2D-LIFlabels/DroneVehicle_test_labels/labels/val'

def split_profile(split, label_dir, rgb_dir, ir_dir):
    labels = {p.stem: p for p in label_dir.glob('*.txt')}
    rgb = {p.stem: p for p in rgb_dir.iterdir() if p.is_file()}
    ir = {p.stem: p for p in ir_dir.iterdir() if p.is_file()}
    lines = [line.strip() for p in labels.values() for line in p.read_text().splitlines() if line.strip()]
    duplicate_extras = sum(sum(count - 1 for count in Counter(
        line.strip() for line in p.read_text().splitlines() if line.strip()
    ).values() if count > 1) for p in labels.values())
    return {
        'split': split, 'rgb': len(rgb), 'ir': len(ir), 'labels': len(labels),
        'raw_boxes': len(lines), 'duplicate_extra_rows': duplicate_extras,
        'stems_match': set(rgb) == set(ir) == set(labels),
        'class_counts': dict(sorted(Counter(int(line.split()[0]) for line in lines).items())),
    }

profiles = [
    split_profile(s, TRAIN_LABEL_ROOT/s, SOURCE_ROOT/'images'/s, SOURCE_ROOT/'image'/s) for s in ('train', 'val')
]
profiles.append(split_profile('test', TEST_LABEL_ROOT, SOURCE_ROOT/'images'/'test', SOURCE_ROOT/'image'/'test'))
pprint(profiles)
d_test_text = (RUNS['D'] / 'test_m2dlif/test.txt').read_text()
for key in ('labels', 'rgb', 'ir', 'class_map', 'split', 'imgsz', 'batch', 'device'):
    match = re.search(rf'^{key}: (.+)$', d_test_text, re.M)
    print(key, '=', match.group(1))

[{'class_counts': {0: 270368, 1: 15834, 2: 11196, 3: 11334, 4: 7701},
  'duplicate_extra_rows': 19,
  'ir': 17990,
  'labels': 17990,
  'raw_boxes': 316433,
  'rgb': 17990,
  'split': 'train',
  'stems_match': True},
 {'class_counts': {0: 20588, 1: 1470, 2: 918, 3: 789, 4: 725},
  'duplicate_extra_rows': 0,
  'ir': 1469,
  'labels': 1469,
  'raw_boxes': 24490,
  'rgb': 1469,
  'split': 'val',
  'stems_match': True},
 {'class_counts': {0: 137162, 1: 8657, 2: 5064, 3: 4467, 4: 4282},
  'duplicate_extra_rows': 14,
  'ir': 8980,
  'labels': 8980,
  'raw_boxes': 159632,
  'rgb': 8980,
  'split': 'test',
  'stems_match': True}]
labels = /media/biiteam/新加卷1/biiteam/MCONG/datasets/M2D-LIFlabels/DroneVehicle_test_labels/labels/val
rgb = /media/biiteam/新加卷1/biiteam/MCONG/datasets/DroneVehicle_twostream_3/images/test
ir = /media/biiteam/新加卷1/biiteam/MCONG/datasets/DroneVehicle_twostream_3/image/test
class_map = M2D-LIF 0->0, 1->1, 2->4, 3->2, 4->3
split = test
imgsz = 640
batch = 16
device = 2


## Results

In [3]:
def read_results(path):
    with path.open() as handle:
        return [{k.strip(): float(v) for k, v in row.items()} for row in csv.DictReader(handle)]

def read_test(path):
    output = {}
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 7 and parts[0] in {'all','car','truck','bus','van','freight_car'}:
            output[parts[0]] = {'instances': int(parts[2]), 'P': float(parts[3]), 'R': float(parts[4]), 'mAP50': float(parts[5]), 'mAP50-95': float(parts[6])}
    return output

curves = {name: read_results(path/'results.csv') for name, path in RUNS.items()}
tests = {name: read_test(path/'test_m2dlif/test.txt') for name, path in RUNS.items()}
metric = 'metrics/mAP50-95(B)'
validation = {}
for name, curve in curves.items():
    best = max(curve, key=lambda row: row[metric])
    validation[name] = {
        'best_epoch': int(best['epoch']), 'best_mAP50-95': best[metric],
        'last30_mean': statistics.mean(row[metric] for row in curve[-30:]),
        'last30_std': statistics.pstdev(row[metric] for row in curve[-30:]),
        'last': curve[-1][metric],
    }
print('test overall')
pprint({name: tests[name]['all'] for name in RUNS})
print('validation')
pprint(validation)
for peer in ('A','B','C'):
    differences = [left[metric] - right[metric] for left, right in zip(curves['D'][-30:], curves[peer][-30:])]
    print('D minus', peer, 'last30 mean', statistics.mean(differences), 'positive epochs', sum(v > 0 for v in differences), '/30')
print('D class mAP50-95 deltas vs A')
pprint({c: tests['D'][c]['mAP50-95'] - tests['A'][c]['mAP50-95'] for c in ('car','truck','bus','van','freight_car')})

test overall
{'A': {'P': 0.799,
       'R': 0.789,
       'instances': 159618,
       'mAP50': 0.819,
       'mAP50-95': 0.667},
 'B': {'P': 0.794,
       'R': 0.793,
       'instances': 159618,
       'mAP50': 0.818,
       'mAP50-95': 0.668},
 'C': {'P': 0.803,
       'R': 0.79,
       'instances': 159618,
       'mAP50': 0.822,
       'mAP50-95': 0.67},
 'D': {'P': 0.805,
       'R': 0.783,
       'instances': 159618,
       'mAP50': 0.813,
       'mAP50-95': 0.664}}
validation
{'A': {'best_epoch': 66,
       'best_mAP50-95': 0.71995,
       'last': 0.71743,
       'last30_mean': 0.7182616666666667,
       'last30_std': 0.0007319748781359983},
 'B': {'best_epoch': 80,
       'best_mAP50-95': 0.72366,
       'last': 0.72164,
       'last30_mean': 0.7227036666666666,
       'last30_std': 0.0006187917438219676},
 'C': {'best_epoch': 71,
       'best_mAP50-95': 0.72181,
       'last': 0.71888,
       'last30_mean': 0.7202223333333333,
       'last30_std': 0.0007115367562927177},
 'D': {

In [4]:
d_best = max(curves['D'], key=lambda row: row[metric])
prompt = {}
for level in ('P3','P4'):
    rgb_recall = d_best[f'p2/{level}_rgb_win_recall']
    ir_recall = d_best[f'p2/{level}_ir_win_recall']
    prompt[level] = {
        'hard_accuracy': d_best[f'p2/{level}_object_prompt_accuracy'],
        'balanced_accuracy': (rgb_recall + ir_recall)/2,
        'correlation': d_best[f'p2/{level}_object_prompt_correlation'],
        'predicted_rgb': d_best[f'p2/{level}_object_prompt_rgb'],
        'target_rgb': d_best[f'p2/{level}_object_target_rgb'],
    }
pprint(prompt)
print('teacher RGB win rate', d_best['p2/teacher_rgb_win_rate'])
print('D P3 residual gain at best epoch', d_best['p2/P3_prompt_residual_gain'])
print('D P3 residual gain last 10 mean', statistics.mean(row['p2/P3_prompt_residual_gain'] for row in curves['D'][-10:]))

{'P3': {'balanced_accuracy': 0.675245,
        'correlation': 0.43061,
        'hard_accuracy': 0.68748,
        'predicted_rgb': 0.43962,
        'target_rgb': 0.40986},
 'P4': {'balanced_accuracy': 0.69085,
        'correlation': 0.46325,
        'hard_accuracy': 0.69225,
        'predicted_rgb': 0.44191,
        'target_rgb': 0.40986}}
teacher RGB win rate 0.26625
D P3 residual gain at best epoch 0.088679
D P3 residual gain last 10 mean 0.0889298


## Takeaways

1. The labels are correct. Train and validation use the M2D-LIF train/val label root and the D evaluator uses the M2D-LIF test label root. RGB, IR, and label stems match exactly. The test evaluator removes 14 duplicate car rows, explaining 159632 raw rows versus 159618 evaluated instances; this applies equally to A/B/C/D.
2. D is a negative result: test mAP50-95 is 0.664 and test mAP50 is 0.813. Precision rises to 0.805 while recall falls to 0.783, indicating a more conservative detector rather than improved localization.
3. The regression is broad: relative to A, truck mAP50-95 falls 0.007 and van falls 0.010; only bus improves materially (+0.003).
4. Prompt learning did not fail. D's P3/P4 balanced accuracy and correlations are slightly better than B/C, and the P3 residual gain reaches about +0.089. The problem is that this learned P3 reliability signal does not transfer into better detection when used alone.
5. B remains the robust choice. Its final-30 validation mean is 0.00358 above D and it beats D in all 30 late epochs. C's small negative P4 residual may be a compensating cross-scale correction, but one non-deterministic seed cannot establish that causally.